In [23]:
import math
from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister, ClassicalRegister
from qiskit.circuit.library import MCXGate


In [24]:
#for convience of the construction we first used QFT with no-swap and build the function
#and since we are only allowed use controlled phase gate and thus the inversion will be used later as well
def qft_no_swaps(qc: QuantumCircuit, reg):
    n = len(reg)
    for j in range(n):
        qc.h(reg[j])
        for k in range(j + 1, n):
            qc.cp(math.pi / (1 << (k - j)), reg[k], reg[j])


def iqft_no_swaps(qc: QuantumCircuit, reg):
    n = len(reg)
    for j in reversed(range(n)):
        for k in reversed(range(j + 1, n)):
            qc.cp(-math.pi / (1 << (k - j)), reg[k], reg[j])
        qc.h(reg[j])


In [25]:
#since the result will be stored in multiple qubits we need to use the MCX to construct the function
def mcx_generic(qc: QuantumCircuit, controls, target):
    qc.append(MCXGate(len(controls)), list(controls) + [target])

In [26]:
#functions in building oracle
def controlled_add_const_in_qft(qc: QuantumCircuit, ctrl, acc, const: int):
    """
    Adds classical constant `const` to `acc` IN THE QFT BASIS, controlled on `ctrl`.
    Assumes acc is little-endian (acc[0]=LSB).
    """
    n = len(acc)
    const %= (1 << n)
    if const == 0:
        return

    for k in range(n):
        angle = 0.0
        for j in range(k + 1):
            if (const >> j) & 1:
                angle += 2 * math.pi / (1 << (k - j + 1))
        if angle != 0.0:
            qc.cp(angle, ctrl, acc[k])


def add_subset_sum_into_acc(qc: QuantumCircuit, subset, acc, xs):
    """
    Compute sum_i subset[i]*xs[i] into acc (acc starts |0>), then return acc to comp basis.
    """
    qft_no_swaps(qc, acc)
    for i, xi in enumerate(xs):
        controlled_add_const_in_qft(qc, subset[i], acc, xi)
    iqft_no_swaps(qc, acc)


def add_const(qc: QuantumCircuit, acc, const: int):
    """
    Uncontrolled add constant to acc (computational basis) using QFT phases.
    """
    m = len(acc)
    const %= (1 << m)
    if const == 0:
        return

    qft_no_swaps(qc, acc)
    for k in range(m):
        angle = 0.0
        for j in range(k + 1):
            if (const >> j) & 1:
                angle += 2 * math.pi / (1 << (k - j + 1))
        if angle != 0.0:
            qc.p(angle, acc[k])
    iqft_no_swaps(qc, acc)


def sub_const(qc: QuantumCircuit, acc, const: int):
    """
    Subtract constant from acc (computational basis).
    """
    m = len(acc)
    neg = ((1 << m) - (const % (1 << m))) % (1 << m)
    add_const(qc, acc, neg)


In [27]:
#oracle construction

def build_subset_sum_phase_oracle(xs, t, m):
    n = len(xs)

    subset = QuantumRegister(n, "s")
    acc = QuantumRegister(m, "a")
    anc = AncillaRegister(max(1, m - 1), "anc")  
    ph = QuantumRegister(1, "ph")

    qc = QuantumCircuit(subset, acc, anc, ph, name="Oracle")
 
    qc.x(ph[0])
    qc.h(ph[0])

    # compute sum into acc
    add_subset_sum_into_acc(qc, subset, acc, xs)

    # acc := acc - t  (sum==t <=> acc==0)
    sub_const(qc, acc, t)

    # mark acc==0 by flipping acc bits then MCX onto phase(since MCX only works when all digits are 1)
    for q in acc:
        qc.x(q)
    mcx_generic(qc, list(acc), ph[0])
    for q in acc:
        qc.x(q)

    # uncompute: add t back
    add_const(qc, acc, t)

    # uncompute subset sum: add (-xi) controlled on subset[i]
    qft_no_swaps(qc, acc)
    for i, xi in enumerate(xs):
        neg_xi = ((1 << m) - (xi % (1 << m))) % (1 << m)
        controlled_add_const_in_qft(qc, subset[i], acc, neg_xi)
    iqft_no_swaps(qc, acc)

    # return phase qubit to |0> (optional)
    qc.h(ph[0])
    qc.x(ph[0])

    return qc

In [28]:
# Construct Diffuser

def build_diffuser(n):
    s = QuantumRegister(n, "s")
    anc = AncillaRegister(max(1, n - 1), "anc_d")
    qc = QuantumCircuit(s, anc, name="Diffuser")

    qc.h(s)
    qc.x(s)

    qc.h(s[n - 1])
    mcx_generic(qc, list(s[:-1]), s[n - 1])
    qc.h(s[n - 1])

    qc.x(s)
    qc.h(s)

    return qc

In [29]:
#applied Grover algorithm(repeat oracle and diffuser)
def subset_sum_quantum(xs, t, iters=None):
    n = len(xs)
    max_sum = sum(xs)
    m = max(1, math.ceil(math.log2(max_sum + 1)))

    if iters is None:
        iters = max(1, int((math.pi / 4) * math.sqrt(2 ** n)))

    subset = QuantumRegister(n, "s")
    acc = QuantumRegister(m, "a")
    anc_or = AncillaRegister(max(1, m - 1), "anc_or")
    ph = QuantumRegister(1, "ph")
    anc_d = AncillaRegister(max(1, n - 1), "anc_d")

    qc = QuantumCircuit(subset, acc, anc_or, ph, anc_d)

    qc.h(subset)

    oracle = build_subset_sum_phase_oracle(xs, t, m).to_gate()
    diffuser = build_diffuser(n).to_gate()

    for _ in range(iters):
        qc.append(oracle, list(subset) + list(acc) + list(anc_or) + list(ph))
        qc.append(diffuser, list(subset) + list(anc_d))

    return qc

In [30]:
#applied alorithm and chose the high probability result
def sample_solutions(xs, t, shots=500, iters=2):
    from qiskit.primitives import StatevectorSampler

    qc = subset_sum_quantum(xs, t, iters=iters)

    n = len(xs)
    c = ClassicalRegister(n, "c_s")
    qc.add_register(c)

    # measure only subset register (first qreg)
    qc.measure(qc.qregs[0], c)

    sampler = StatevectorSampler()
    result = sampler.run([qc], shots=shots).result()

    counts = result[0].data.c_s.get_counts()

    best = max(counts, key=counts.get)

    # Qiskit prints MSB..LSB; reverse to map to s_0..s_{n-1}
    s_bits = best[::-1]
    I = {i for i, b in enumerate(s_bits) if b == "1"}

    return counts, best, s_bits, I


In [36]:
#for better result print and solve the case for no result
def solve_subset_sum(xs, t, shots=500, attempts=6, iters_list=None, dominance_threshold=0.20):
    """
    Returns:
      - (I, info_dict) if a correct solution is found (verified classically).
      - (None, info_dict) if no solution is found after several attempts
        (interpreted as "no solution with high probability").

    Strategy:
      - Try several Grover iteration counts (Grover oscillates).
      - Each time: run sample_solutions -> get candidate I.
      - Verify classically: sum(xs[i] for i in I) == t.
      - If none verify, return None.
    """
    n = len(xs)
    if n == 0:
        return None, {"reason": "empty input"}

    # Default schedule: try around the usual Grover heuristic sqrt(2^n)
    if iters_list is None:
        base = max(1, int((math.pi / 4) * math.sqrt(2 ** n)))
        iters_list = sorted(set([1, 2, 3, max(1, base - 1), base, base + 1, max(1, base // 2)]))
        iters_list = iters_list[:attempts]

    best_seen = None  # store best dominance attempt for reporting

    for iters in iters_list:
        counts, best, s_bits, I = sample_solutions(xs, t, shots=shots, iters=iters)
        candidate_sum = sum(xs[i] for i in I)

        dominance = counts[best] / shots
        if best_seen is None or dominance > best_seen["dominance"]:
            best_seen = {
                "iters": iters,
                "best": best,
                "s_bits": s_bits,
                "I": I,
                "dominance": dominance,
                "candidate_sum": candidate_sum,
            }

        # Verified success
        if candidate_sum == t:
            return I, {
                "status": "FOUND",
                "iters": iters,
                "best_bitstring": best,
                "dominance": dominance,
                "s_bits": s_bits,
                "I": I,
                "candidate_sum": candidate_sum,
                "shots": shots,
            }

    # Nothing verified => "no solution with high probability"
    # dominance_threshold is just an extra signal; we still return None regardless.
    return None, {
        "status": "NO_SOLUTION_WHP",
        "shots": shots,
        "attempts": len(iters_list),
        "iters_list": iters_list,
        "best_seen": best_seen,
        "note": (
            "No measured candidate verified. If no true solution exists, Grover cannot amplify, "
            "so outcomes remain near-uniform; repeated failures support 'no solution' conclusion."
        ),
    }

In [32]:
if __name__ == "__main__":
    xs = [3, 5, 6, 2]
    t = 11

    I, info = solve_subset_sum(xs, t, shots=500, attempts=6)

    print("INFO:", info)
    if I is None:
        print("No solution found (high probability).")
    else:
        print("Solution I:", I)
        print("Check sum:", sum(xs[i] for i in I))

INFO: {'status': 'FOUND', 'iters': 1, 'best_bitstring': '1101', 'dominance': 0.402, 's_bits': '1011', 'I': {0, 2, 3}, 'candidate_sum': 11, 'shots': 500}
Solution I: {0, 2, 3}
Check sum: 11


In [34]:
if __name__ == "__main__":
    xs = [2, 4, 6, 8]
    t = 10

    counts, best, s_bits, I = sample_solutions(xs, t, shots=500, iters=2)

    print("Best bitstring:", best)
    print("Subset bits s0..:", s_bits)
    print("Index set I:", I)
    print("Check sum:", sum(xs[i] for i in I))

Best bitstring: 1001
Subset bits s0..: 1001
Index set I: {0, 3}
Check sum: 10


In [38]:
if __name__ == "__main__":
    xs = [2, 4, 6, 8]
    t = 10

    I, info = solve_subset_sum(xs, t, shots=500, attempts=6)

    print("INFO:", info)
    if I is None:
        print("No solution found (high probability).")
    else:
        print("Solution I:", I)
        print("Check sum:", sum(xs[i] for i in I))

INFO: {'status': 'FOUND', 'iters': 1, 'best_bitstring': '1001', 'dominance': 0.382, 's_bits': '1001', 'I': {0, 3}, 'candidate_sum': 10, 'shots': 500}
Solution I: {0, 3}
Check sum: 10


In [ ]:
#bench mark setting
import random
from collections import defaultdict

#the brute-force method for verifying 
def brute_force_has_solution(xs, t):
    n = len(xs)
    for mask in range(1 << n):
        s = 0
        for i in range(n):
            if (mask >> i) & 1:
                s += xs[i]
        if s == t:
            return True
    return False

#benchmark
def benchmark(trials=200, n=4, xi_max=7, shots=500, attempts=6, seed=0):
    random.seed(seed)
    stats = defaultdict(int)

    for _ in range(trials):
        xs = [random.randint(1, xi_max) for _ in range(n)]
        t = random.randint(1, sum(xs))

        has_solution = brute_force_has_solution(xs, t)

        I, info = solve_subset_sum(xs, t, shots=shots, attempts=attempts)

        if has_solution:
            stats["with_solution"] += 1
            if I is None:
                stats["missed"] += 1
            else:
                # verified inside solve_subset_sum
                stats["found"] += 1
        else:
            stats["no_solution"] += 1
            if I is None:
                stats["correct_no_solution"] += 1
            else:
                stats["false_positive"] += 1

    print("=== Benchmark n<=4, xi<8 ===")
    print(f"trials={trials}, n={n}, xi_max={xi_max}, shots={shots}, attempts={attempts}, seed={seed}")
    if stats["with_solution"]:
        print("With-solution:")
        print("  found rate:", stats["found"] / stats["with_solution"])
        print("  missed rate:", stats["missed"] / stats["with_solution"])
    if stats["no_solution"]:
        print("No-solution:")
        print("  correct no-solution rate:", stats["correct_no_solution"] / stats["no_solution"])
        print("  false positive rate:", stats["false_positive"] / stats["no_solution"])


if __name__ == "__main__":
    benchmark()

=== Benchmark n<=4, xi<8 ===
trials=200, n=4, xi_max=7, shots=500, attempts=6, seed=0
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0


In [ ]:
#since n is small and the shots are large we have a oretty reliable result

In [41]:
def benchmark(trials=200, n=4, xi_max=7, shots=10, attempts=6, seed=0):
    random.seed(seed)
    stats = defaultdict(int)

    for _ in range(trials):
        xs = [random.randint(1, xi_max) for _ in range(n)]
        t = random.randint(1, sum(xs))

        has_solution = brute_force_has_solution(xs, t)

        I, info = solve_subset_sum(xs, t, shots=shots, attempts=attempts)

        if has_solution:
            stats["with_solution"] += 1
            if I is None:
                stats["missed"] += 1
            else:
                # verified inside solve_subset_sum
                stats["found"] += 1
        else:
            stats["no_solution"] += 1
            if I is None:
                stats["correct_no_solution"] += 1
            else:
                stats["false_positive"] += 1

    print("=== Benchmark n<=4, xi<8 ===")
    print(f"trials={trials}, n={n}, xi_max={xi_max}, shots={shots}, attempts={attempts}, seed={seed}")
    if stats["with_solution"]:
        print("With-solution:")
        print("  found rate:", stats["found"] / stats["with_solution"])
        print("  missed rate:", stats["missed"] / stats["with_solution"])
    if stats["no_solution"]:
        print("No-solution:")
        print("  correct no-solution rate:", stats["correct_no_solution"] / stats["no_solution"])
        print("  false positive rate:", stats["false_positive"] / stats["no_solution"])


if __name__ == "__main__":
    benchmark()

=== Benchmark n<=4, xi<8 ===
trials=200, n=4, xi_max=7, shots=10, attempts=6, seed=0
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0


In [ ]:
for seed in range(10):
    benchmark(trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=seed)

=== Benchmark n<=4, xi<8 ===
trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=0
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0
=== Benchmark n<=4, xi<8 ===
trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=1
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0
=== Benchmark n<=4, xi<8 ===
trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=2
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0
=== Benchmark n<=4, xi<8 ===
trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=3
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-solution rate: 1.0
  false positive rate: 0.0
=== Benchmark n<=4, xi<8 ===
trials=500, n=4, xi_max=7, shots=10, attempts=6, seed=4
With-solution:
  found rate: 1.0
  missed rate: 0.0
No-solution:
  correct no-s